# Foundry IQ v2: 공정 이해와 품질 후속 조치

**목표:** 공정 KB 3개 KS와 품질 KB 4개 KS를 각각 조회하고, 답변의 사실·출처·미확인 항목을 실행 로그로 확인합니다. 처음부터 끝까지 v2만 사용합니다.

**실행 순서:** 설치 → 1. 설정 → 2. MSAL 사용자 로그인 → 3. 호출·로그 준비 → 4. 공정 KB → 5. 품질 KB. 각 질문 호출 셀을 실행할 때마다 KB를 조회하고 모델·소스 사용량이 발생합니다. 로그 출력 셀만 다시 실행하면 추가 조회 비용은 없습니다.

## 사전 준비

- 설정한 Search 서비스에 v2 KB 두 개와 연결된 KS가 준비되어 있어야 합니다. 이 노트북은 리소스를 생성·변경·삭제하지 않습니다.
- 로그인할 테넌트의 사용자 계정과 앱 등록을 준비합니다. **Azure CLI 로그인이나 클라이언트 비밀키는 사용하지 않습니다.**
- `WORKIQ_APP_ID` 앱의 Mobile and desktop applications 플랫폼에 `http://localhost` 리디렉션 URI를 등록합니다.
- 같은 앱의 **API permissions → Add a permission → Azure Cognitive Search → Delegated permissions → user_impersonation**을 추가하고 필요한 동의를 완료합니다. 누락하면 Search 토큰 발급 시 `AADSTS650057`이 발생할 수 있습니다. 기존 Work IQ 권한은 유지합니다.
- Work IQ 앱에 `access_as_user` scope와 소스 연결에 필요한 설정이 준비되어 있어야 합니다. 직접 Work IQ REST용 `WorkIQAgent.Ask` 토큰과는 다릅니다.
- 로그인 사용자는 Search Index Data Reader와 원본 M365·Fabric 접근 권한이 필요합니다. 앱의 API 권한과 Search 리소스의 사용자 역할은 별개입니다.
- Fabric capacity는 실행 중이어야 합니다. MES·Web IQ 키는 각 KS의 연결 설정에 준비하며, 이 노트북에서 사용자 토큰으로 대체하지 않습니다.

## 사용자 토큰 전달

| 헤더 | 값 | 전달 대상 |
| --- | --- | --- |
| `Authorization` | Search 대상 사용자 토큰 | 공정·품질 KB |
| `x-ms-query-source-authorization` | Search 대상 사용자 토큰 | 품질 KB의 Fabric |
| `x-ms-query-work-iq-source-authorization` | `api://<앱 ID>/access_as_user` 사용자 토큰 | 품질 KB의 Work IQ |

현재 Work IQ KS는 포털의 KB 테스트 화면에서 지원되지 않습니다. 이 노트북은 REST 요청에 필요한 사용자 헤더를 전달합니다. [Work IQ KS 공식 안내](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-work-iq)

**남아 있는 저장 출력은 이전 검증 이력입니다.** 현재 MSAL 로그인이나 새 실행의 성공을 증명하지 않습니다. 공정 기본 질문을 LOT0012 조사로 변경하면서 이전 공정 답변·로그 출력은 제거했습니다. 본인 환경은 커널을 재시작하고 위에서부터 실행해 확인하세요. [실습 안내](README.md) · [검증 이력](helper/code/v2_validation.json)


In [2]:
%pip install --quiet requests msal


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. 설정

자신의 Search 엔드포인트·테넌트·Work IQ 앱 ID를 확인합니다. KB 이름은 공정·품질을 각각 지정하며, 연결된 KS 이름은 3절의 `V2_SOURCES`와 일치해야 합니다. 환경 변수를 사용하지 않으면 아래 기본값을 직접 수정하세요.


In [3]:
import os

SEARCH_ENDPOINT = os.getenv("SEARCH_ENDPOINT", "https://ai-search-srch-vljt.search.windows.net")
PROCESS_KB_NAME = os.getenv("PROCESS_KB_NAME", "process-assistance-kb-v2")
QUALITY_KB_NAME = os.getenv("QUALITY_KB_NAME", "quality-investigation-kb-v2")
TENANT_ID = os.getenv("TENANT_ID", "b6dd05dd-72f3-49f5-80e4-866e3a943565")
WORKIQ_APP_ID = os.getenv("WORKIQ_APP_ID", "12beac79-cff2-47e0-9fc3-4b60e3b33540")
API_VERSION = "2026-08-01-preview"

settings = {
    "SEARCH_ENDPOINT": SEARCH_ENDPOINT,
    "PROCESS_KB_NAME": PROCESS_KB_NAME,
    "QUALITY_KB_NAME": QUALITY_KB_NAME,
    "TENANT_ID": TENANT_ID,
    "WORKIQ_APP_ID": WORKIQ_APP_ID,
}
missing = [name for name, value in settings.items() if not value or "<" in value]
if missing:
    raise ValueError(f"다음 설정값을 채우세요: {', '.join(missing)}")
if PROCESS_KB_NAME == QUALITY_KB_NAME:
    raise ValueError("공정 KB와 품질 KB는 서로 다른 이름이어야 합니다.")

print("공정 KB:", PROCESS_KB_NAME)
print("품질 KB:", QUALITY_KB_NAME)


공정 KB: process-assistance-kb-v2
품질 KB: quality-investigation-kb-v2


## 2. MSAL 사용자 로그인

브라우저에서 실습 계정으로 로그인합니다. Work IQ 사용자 assertion을 발급한 뒤 같은 MSAL 앱으로 Search 토큰을 요청합니다. Search 토큰은 캐시에서 먼저 시도하고 필요한 경우 브라우저 인증이 추가됩니다.

`Search 토큰, Work IQ 토큰 발급 완료`를 확인한 뒤 다음 절로 진행합니다. `AADSTS650057`이면 1절의 앱 ID에 Azure Cognitive Search 위임 권한과 필요한 동의가 등록됐는지 확인하고, 설정 후 커널을 재시작해 다시 실행하세요. 토큰·인증 헤더는 공유하지 않습니다.


In [4]:
import msal

WORKIQ_SCOPES = [f"api://{WORKIQ_APP_ID}/access_as_user"]
SEARCH_SCOPES = ["https://search.azure.com/.default"]

auth_app = msal.PublicClientApplication(
    WORKIQ_APP_ID, authority=f"https://login.microsoftonline.com/{TENANT_ID}"
)

def check_token(result, name):
    if "access_token" not in result:
        raise RuntimeError(
            f"{name} 토큰 발급 실패: {result.get('error', 'unknown')} | "
            f"오류 코드: {result.get('error_codes', [])} | "
            f"correlation_id: {result.get('correlation_id', '(없음)')}\n"
            f"{result.get('error_description', '')}"
        )
    return result["access_token"]


def login(scopes, **kwargs):
    try:
        return auth_app.acquire_token_interactive(scopes=scopes, timeout=300, **kwargs)
    except msal.BrowserInteractionTimeoutError as exc:
        raise RuntimeError(
            "로그인이 5분 안에 완료되지 않았습니다. 브라우저 팝업 차단, 기본 브라우저 설정, "
            "네트워크/프록시 상태를 확인한 뒤 이 셀을 다시 실행하세요."
        ) from exc


workiq_login = login(WORKIQ_SCOPES, prompt="select_account")
workiq_token = check_token(workiq_login, "Work IQ")

account = auth_app.get_accounts()[0]
search_login = auth_app.acquire_token_silent(SEARCH_SCOPES, account=account)
if not search_login or "access_token" not in search_login:
    search_login = login(SEARCH_SCOPES, login_hint=account["username"])
    
search_token = check_token(search_login, "Search")

print(f"로그인 계정: {account['username']}")
print("Search 토큰, Work IQ 토큰 발급 완료")

로그인 계정: admin@m365cpi65716313.onmicrosoft.com
Search 토큰, Work IQ 토큰 발급 완료


## 3. KB 호출과 로그 파싱 준비

로그인 후 아래 준비 셀을 실행합니다. 이 셀은 도우미를 불러오고 호출 함수를 정의하며 API를 호출하지 않습니다. 아래 각 v2 질문 호출에는 모델·소스 사용량이 발생하지만 **로그 파싱과 재표시는 추가 호출·비용이 없습니다.** Fabric 용량이 Active이고 사용자에게 원본 접근 권한이 있어야 합니다.

`includeActivity`, `includeReferences`, `includeReferenceSourceData`로 받은 JSON을 직접 파싱합니다. **모델의 내부 추론 전문은 반환되지 않습니다.** 계획 활동·토큰, 실제 하위 검색어·필터·도구 입력, 결과 수, 경고·오류와 인용 연결을 확인합니다. 재조회 이유가 응답에 없으면 추측하지 않습니다.

1. **타임라인:** 시작·종료 시각과 겹치는 활동 ID. 병렬 활동 시간을 합산하지 않습니다.
2. **검색 상세:** 소스별 `*Arguments`를 그대로 보여 질문이 어떤 실제 호출로 나뉘었는지 확인합니다.
3. **동일 소스 추가 호출:** 이전·후속 입력과 건수를 대조합니다. 병렬 검색도 포함될 수 있으며, 계획 활동 반복만으로 재검색·재시도로 판정하지 않습니다.
4. **인용 연결:** `activity.id → references.activitySource → reference.id → [ref_id:…]`. `count`는 관련성 기준을 통과한 결과 수이며 원본 전체 건수나 최종 인용 수가 아닙니다.

출력에 문서·메일 발췌가 포함될 수 있습니다. 공유 전 출력 내용을 검토하고 필요하면 지우세요. 키·토큰은 출력하지 않습니다. 공정 질문에는 공개 가능한 기술 용어만 사용합니다. KB 지침은 보안 필터가 아닙니다.

[공식 retrieve 응답 계약](https://learn.microsoft.com/en-us/rest/api/searchservice/knowledge-retrieval/retrieve?view=rest-searchservice-2026-08-01-preview)

In [5]:
from pathlib import Path
import sys
import time

import requests
from IPython.display import Markdown, display

helper_dir = next((parent / 'labs/05-foundry-iq/helper/code'
                   for parent in [Path.cwd(), *Path.cwd().parents]
                   if (parent / 'labs/05-foundry-iq/helper/code/retrieval_trace.py').is_file()), None)
if helper_dir is None:
    raise FileNotFoundError('저장소 안에서 실행하고 helper/code/retrieval_trace.py가 있는지 확인하세요.')
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
from retrieval_trace import parse_trace, format_trace
from knowledge_bases_v2 import has_evidence

V2_SOURCES = {
    PROCESS_KB_NAME: {
        'ks-process-mes-v2': 'mcpServer',
        'ks-process-webiq-v2': 'mcpServer',
        'ks-quality-manuals-v2': 'azureBlob',
    },
    QUALITY_KB_NAME: {
        'ks-quality-handovers-v2': 'searchIndex',
        'ks-quality-workorders-v2': 'searchIndex',
        'ks-quality-fabric-v2': 'fabricDataAgent',
        'ks-quality-workiq-v2': 'workIQ',
    },
}

def retrieve_v2_trace(kb_name, question):
    headers = {'Authorization': f'Bearer {search_token}'}
    if kb_name == QUALITY_KB_NAME:
        headers['x-ms-query-source-authorization'] = search_token
        headers['x-ms-query-work-iq-source-authorization'] = workiq_token
    started = time.monotonic()
    response = requests.post(
        f'{SEARCH_ENDPOINT}/knowledgebases/{kb_name}/retrieve',
        params={'api-version': API_VERSION}, headers=headers,
        json={
            'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}],
            'knowledgeSourceParams': [
                {'knowledgeSourceName': name, 'kind': kind,
                 'includeReferences': True, 'includeReferenceSourceData': True}
                for name, kind in V2_SOURCES[kb_name].items()
            ],
            'includeActivity': True, 'maxRuntimeInSeconds': 300,
        }, timeout=330,
    )
    print(f'{kb_name}: HTTP {response.status_code} / {time.monotonic() - started:.1f}초')
    response.raise_for_status()
    result = response.json()
    if response.status_code == 206:
        print('부분 실패: 생성형 답변보다 activity.error를 우선 확인하세요.')
    for name in V2_SOURCES[kb_name]:
        print(name, '근거 수신' if has_evidence(result, name) else '미확인 또는 실패')
    return result

print('로그 파서 준비 완료. 이 셀은 API를 호출하지 않습니다.')

로그 파서 준비 완료. 이 셀은 API를 호출하지 않습니다.


## 4. 공정 KB: MES·내부 매뉴얼·Web IQ

LOT0012의 상태에서 시작해 이슈가 있다면 직전 공정 설비·내부 문서·공개 사례를 조사합니다. 질문은 아래 문장 그대로 사용합니다. 실제 이력 연결, 원인 가설과 확인 사실의 구분, 내부 정보의 외부 검색 제외는 KB 지침과 처리에서 책임질 사항입니다.

호출·로그 셀에서 실제로 선택된 소스와 근거를 확인하세요. 이슈가 확인되지 않아 후속 소스를 조회하지 않은 경우와, 필요한 근거를 누락한 경우를 구분합니다. HTTP 206 또는 소스 오류는 부분 실패입니다. MES만으로 QMS 검사·FDC 센서 이상까지 확인했다고 해석하지 않습니다.


In [ ]:
v2_process_question = (
    'MES에서 LOT0012 로트 상태를 확인하고, 현재 진행 중인 공정에서 이슈가 있는지 확인해보자. '
    '만약 이슈가 있다면, 바로 직전 공정의 설비를 확인해서 해당 설비를 내부 장비 문서를 통해서 '
    '어떤 이슈가 있을법한지 검토해 보고, 이 이슈 사항이 웹 상에서 다른 회사에서도 문제가 '
    '발생했는지 퍼블릭하게 확인해서 전체적인 요약본을 제공해'
)
v2_process_result = retrieve_v2_trace(PROCESS_KB_NAME, v2_process_question)

In [ ]:
v2_process_trace = parse_trace(v2_process_result)
print(format_trace(v2_process_trace))
display(Markdown('### 최종 답변\n' + v2_process_trace['answer']))

In [ ]:
v2_process_question = (
    'MES에서 제품창고에서 대기 중인 로트를 3개를 MES에서 조회 후 리스트화하고, '
    '불량이 발생하여 투입수량과 제품창고 수량이 다른 로트를 찾은 뒤 '
    
)
v2_process_result = retrieve_v2_trace(PROCESS_KB_NAME, v2_process_question)

In [ ]:
v2_process_trace = parse_trace(v2_process_result)
print(format_trace(v2_process_trace))
display(Markdown('### 최종 답변\n' + v2_process_trace['answer']))

## 5. 품질 KB: Fabric·작업지시서·인수인계서·Work IQ

아래 호출은 품질 KB에 한 번 질문합니다. 네 KS 모두 `근거 수신`인지 확인한 뒤 운영 상태·문서 이력·협업 요청을 구분합니다. 같은 로트·설비라는 이유만으로 다른 시점의 사건을 합치거나, 요청을 승인·완료로 해석하지 않습니다.


In [6]:
v2_quality_question = (
    'LOT0004 / EQP-CMP01 / NCR-2026-0009의 후속 조치를 요약하세요. '
    'Fabric에서는 해당 NCR 한 건의 현재 상태를 확인하고, 작업지시서와 교대 인수인계서에서는 '
    '관련 조치 이력을, Work IQ의 IQ-DEMO-20260922 메일에서는 요청·결정·담당자를 확인하세요. '
    '네 출처를 구분하고 없는 결정은 추측하지 마세요.'
)
v2_quality_result = retrieve_v2_trace(QUALITY_KB_NAME, v2_quality_question)

quality-investigation-kb-v2: HTTP 200 / 113.5초
ks-quality-handovers-v2 근거 수신
ks-quality-workorders-v2 근거 수신
ks-quality-fabric-v2 근거 수신
ks-quality-workiq-v2 근거 수신


In [6]:
v2_quality_trace = parse_trace(v2_quality_result)
print(format_trace(v2_quality_trace))
display(Markdown('### 최종 답변\n' + v2_quality_trace['answer']))

1. 실행 타임라인 (내부 추론 전문이 아닌 API 실행 기록)
시작 시각순입니다. 시작 시각 미제공 활동은 뒤에 응답 순서대로 표시합니다.
ID | 활동 | KS | 시작 → 종료 | 소요 ms | 관련 결과 수
0 | modelQueryPlanning | 미제공 | 2026-10-01T01:26:46.6506873Z → 2026-10-01T01:26:53.2965753Z | 6645 (elapsedMs) | 미제공
{
  "inputTokens": 2605,
  "outputTokens": 421,
  "model": {
    "modelName": "gpt-5.2",
    "deploymentId": "gpt-5.2"
  }
}
1 | fabricDataAgent | ks-quality-fabric-v2 | 2026-10-01T01:26:53.2969277Z → 2026-10-01T01:28:16.3904153Z | 83093.488 (startedAt/completedAt 차이) | 1
{
  "queryTime": "2026-10-01T01:26:53.2970572Z"
}
2 | searchIndex | ks-quality-workorders-v2 | 2026-10-01T01:26:53.2969306Z → 2026-10-01T01:26:56.0849106Z | 0 (elapsedMs) | 7
{
  "queryTime": "2026-10-01T01:26:54.0787137Z"
}
3 | searchIndex | ks-quality-workorders-v2 | 2026-10-01T01:26:54.2325949Z → 2026-10-01T01:26:56.0849126Z | 0 (elapsedMs) | 20
{
  "queryTime": "2026-10-01T01:26:54.9964465Z"
}
4 | searchIndex | ks-quality-workorders-v2 | 2026-10-01T01:26:55.0991506Z → 2026-10-01T01:

### 최종 답변
아래는 **LOT0004 / EQP-CMP01 / NCR-2026-0009**의 “후속 조치”를 **네 출처(Fabric / 작업지시서 / 교대 인수인계서 / Work IQ 메일)**로 구분해, 확인된 사실만 요약한 것입니다(없는 결정은 추측하지 않음).

---

## 1) Fabric(운영/QMS 데이터) — NCR 현재 상태 및 후속 품질결정
- NCR `NCR-2026-0009`는 Lot `LOT0004`, 설비 `EQP-CMP01(Polisher)`, 공정 `CMP(Planarization)`에 연결되어 있습니다.[ref_id:0]  
- NCR 상태는 `접수`이며, **종결(Closed) 일자/기록이 없고(Ncr_closed_date = null)**, 현재까지 “완료/종결”로 바뀐 근거가 확인되지 않습니다.[ref_id:0]  
- 검출/기한: `2026-06-20`에 검출되었고, Due date는 `2026-06-27`로 기록돼 있습니다.[ref_id:0]  
- 등급/출처/불량: Severity `Minor`, Source `공정검사`, Defect_code `DEF-OVL-004(Overlay)`로 기록돼 있습니다.[ref_id:0]  
- 영향/비용: 영향 수량 `10`, 추정비용 `900,000 KRW`로 기록돼 있습니다.[ref_id:0]  
- 원인 및 즉시조치(기록된 문구): Root cause category `자재`, 원인 설명 `입고 자재 로트 간 편차가 공정 결과로 전이됨`, 즉시조치 `동일 공급 로트 전량 격리 및 대체 로트 투입`이 NCR 필드에 기록돼 있습니다.[ref_id:0]  
- 담당(Owner): `남유진 / 생산관리팀`으로 기록돼 있습니다.[ref_id:0]  

### (Fabric 내 연계: 검사/판정 근거)
- `LOT0004`가 `EQP-CMP01`에서 `CMP`로 처리된 Run(`Run_id 31`)이 `2026-06-20T03:50:00Z`~`04:45:00Z`로 기록돼 있습니다(UTC).[ref_id:0]  
- 이후 IPQC 검사 `INS-2026-0031`가 `2026-06-20T06:36:00Z`(UTC)에 수행되었고, 판정은 `조건부합격`이며 “후속 공정 모니터링 조건부 승인” 취지의 근거 문구가 기록돼 있고, 이 검사가 NCR과 연결(`Insp_has_nonconformance=true`)돼 있습니다.[ref_id:0]  

### (Fabric 내 연계: Disposition = 후속 품질결정/실행 및 효과 확인)
- NCR에 연결된 Disposition은 `DSP-2026-0009`이며, Lot `LOT0004` / NCR `NCR-2026-0009`로 연결돼 있습니다.[ref_id:0]  
- Disposition_type은 `선별`, Disposition_qty는 `2`, 사유는 `Minor 등급 부적합에 대한 선별 결정`으로 기록돼 있습니다.[ref_id:0]  
- 승인 상태는 `승인`, 승인자 `봉태식`, 의사결정 주체 `생산책임자`, 결정일 `2026-06-21T00:00:00Z`(UTC)로 기록돼 있습니다.[ref_id:0]  
- 효과 확인은 `2026-07-04T00:00:00Z`(UTC)에 수행되었고 결과는 `유효`로 기록돼 있습니다.[ref_id:0]  
- 재작업(rework) 관련 필드는 공정/결과가 비어 있어(Null) **재작업 수행을 확인할 근거가 없습니다**.[ref_id:0]  

---

## 2) 작업지시서(Work Orders) — 관련 조치 이력(문서 기록)
> 주의: 검색된 작업지시서는 문서 상단에 **“워크숍용 합성 자료… 실제 운영 사실이 아니며… 추가 검수가 필요”**라고 명시돼 있습니다.[ref_id:1]  
> 따라서 아래는 “문서에 적힌 조치 이력”을 그대로 요약하되, 실제 이행 여부는 별도 검증이 필요합니다.[ref_id:1]

- 작업지시 `WO-2026-0006`은 설비 `EQP-CMP01`, Lot `LOT0004`, NCR `NCR-2026-0009`, 불량 `DEF-OVL-004(Overlay)`를 직접 언급합니다.[ref_id:1]  
- 문서에 기록된 관찰/센서: FDC 센서 `AMBIENT_TEMP`, 구간 `2026-06-11T13:00:00Z`, 측정 범위 `18.7589~20.1016 degC`, Alarm `17회`가 언급됩니다.[ref_id:1]  
- 문서에 기록된 조치(요약): 배기 댐퍼 상태 확인, 필요 지점 예비품 투입(1개), 이젝터 니들 구간 세정 후 화면 응답 확인, 라인 필터 설정을 표준값으로 복귀, 설비/품질 담당자가 함께 화면 확인, 재가동을 “확인 recipe → hold 조건 해제 → 교대 인수인계” 순서로 마무리했다고 기록돼 있습니다.[ref_id:1]  
- 조치 후 확인(문서 기록): AMBIENT_TEMP 재추적에서 “이전 패턴과 다른 안정 구간 확인”, `LOT0004` 또는 후속 wafer의 MES hold 조건을 “품질보증팀 확인 후 닫았다”고 기록돼 있습니다.[ref_id:1]  

**문서 간 시간 정합성 주의(미확인/충돌 가능):** 위 작업지시서의 날짜는 `2026-06-11`로 기재돼 있는데[ref_id:1], Fabric의 NCR 검출일은 `2026-06-20`으로 기록돼 있어[ref_id:0], 동일 사건의 실제 타임라인인지(또는 사전 점검/다른 이벤트인지) Fabric·원문 결재기록으로 추가 확인이 필요합니다.[ref_id:1][ref_id:0]

---

## 3) 교대 인수인계서(Handovers) — 현장 관찰/조치/다음 조 요청
> 주의: 아래 인수인계서들 또한 **“워크숍용 합성 자료… 실제 운영 사실이 아니며… 추가 검수가 필요”**라고 명시돼 있습니다.[ref_id:2][ref_id:8]

- 인수인계 `SH-2026-0088`은 `EQP-CMP01`, `LOT0004`, `NCR-2026-0009`, `DEF-OVL-004`, 등급 `Minor`를 직접 언급합니다.[ref_id:2]  
  - 인계 시각은 `06:00 (KST)`로 기록돼 있습니다.[ref_id:2]  
  - 관찰: “pad 표면과 ambient temp 저하”가 같은 교대에 남아 있어 “작은 변동의 합”으로 본다는 취지로 기록돼 있습니다.[ref_id:2]  
  - 해당 조 조치: “hold 판단은 보류하고 첫 wafer 결과만 빠르게 공유”, AMBIENT_TEMP 기준값 유지, `EQP-CMP01` recipe 임의 변경 없음으로 기록돼 있습니다.[ref_id:2]  
  - 다음 조 요청: 인수 후 장비 소리 확인 후 이상 시 엔지니어 호출, 대상 lot은 `LOT0004`, `06:00` 이후 첫 확인 결과만 간단히 남길 것을 요청합니다.[ref_id:2]  
  - 사후 기록(관찰 시점과 구분된 ‘사후’): `NCR-2026-0009`가 minor로 접수되었고 CMP 저온/패드 메모가 타임라인에 붙었으며 `CMP-E115` 예방점검을 추가했다고 기록돼 있습니다.[ref_id:2]  

- 인수인계 `SH-2026-0004`는 `EQP-CMP01`, `LOT0004`, 매뉴얼 코드 `CMP-E115`를 언급하며, **사후 기록에 `NCR-2026-0009`가 명시**됩니다.[ref_id:8]  
  - 인계 시각은 `06:00 (KST)`로 기록돼 있습니다.[ref_id:8]  
  - 관찰: `PAD_PRESSURE 37.4 kPa`(Warn 37.8 근처), “헤드가 내려갈 때 진동이 손끝에 더 옴”, 엔드포인트 소리가 늦게 꺼진다는 정성 관찰이 기록돼 있습니다.[ref_id:8]  
  - 해당 조 조치: 화이트보드에 센서명과 재확인 시간을 적어둠, 기준값 유지 및 recipe 임의 변경 없음이 기록돼 있습니다.[ref_id:8]  
  - 다음 조 요청: 로트 교체 직후 2장 집중 관찰 후 정상화되면 진행 가능, 대상 lot `LOT0004`, `06:00` 이후 첫 확인 결과만 남길 것을 요청합니다.[ref_id:8]  
  - 사후 기록: `NCR-2026-0009 LOT0004 Overlay` 리뷰에서 `CMP-E107` 의심 이력으로 채택되었고, pad pressure 수치보다 “head 진동 코멘트”가 결정적이었다고 기록돼 있습니다.[ref_id:8]  

---

## 4) Work IQ (메일: IQ-DEMO-20260922) — 요청·결정·담당자
- Work IQ 조회 결과, **`IQ-DEMO-20260922` 메일 원문 및 그 안의 요청/결정/담당자 정보를 확인할 수 있는 검색 결과가 반환되지 않았습니다.**[ref_id:7]  
- 따라서 **메일 기반의 “요청·결정·담당자”는 이 답변에서 확인 불가**이며, 존재/내용을 추측해 보완하지 않습니다.[ref_id:7]  

---

## (요약) 확인된 후속 조치 & 남은 사항(추측 없음)
- **확인된 후속 조치(결정/실행):** `DSP-2026-0009`에서 `선별(2개)`이 `승인(봉태식/생산책임자)`되었고[ref_id:0], `2026-07-04` 효과 확인 결과가 `유효`로 기록돼 있습니다.[ref_id:0]  
- **즉시조치(기록):** “동일 공급 로트 전량 격리 및 대체 로트 투입”이 NCR 필드에 기록돼 있습니다.[ref_id:0]  
- **남은 사항(상태 관점):** NCR 상태가 여전히 `접수`이고 Closed date가 없어, **종결 처리(또는 상태 갱신) 여부는 미완료로 보입니다(단, 시스템 미반영 가능성은 별도 확인 필요).**[ref_id:0]  
- **다음 조 확인 요청(인수인계서 기록):** 장비 이상 시 엔지니어 호출, 로트 교체 직후 2장 집중 관찰, 06:00 이후 첫 확인 결과 기록 등의 요청이 문서에 남아 있습니다.[ref_id:2][ref_id:8]  
- **Work IQ 메일 기반 의사결정/담당자:** 확인 불가.[ref_id:7]  

원하시면, **NCR 종결이 ‘접수’로 남아 있는 이유(예: 상태 업데이트 누락/잔여 승인/후속 8D 미첨부 등)**를 확인하기 위해, Work IQ 쪽은 **메일 제목/보낸이/받는이 또는 스레드 키워드(예: NCR-2026-0009, LOT0004, DSP-2026-0009)**를 추가로 주시면 그 범위 내에서만 재조회해 드리겠습니다.[ref_id:7]

### 로그를 읽고 확인하기

- `modelQueryPlanning`은 몇 번인가요? 그 사이 실제 검색이 있었나요? 계획의 입·출력 토큰 수가 검색어 텍스트나 추론 전문을 뜻하지는 않습니다.
- 같은 KS의 `*Arguments`가 어떻게 달라졌나요? 병렬 하위 검색과 후속 호출을 시작·종료 시각으로 구분하되, 인과관계가 반환되지 않으면 단정하지 않습니다.
- `warning`의 점수 임계값·토큰 제한·시간 제한 경고, `error`의 실패와 `count=0`을 구분합니다. 타임아웃을 데이터 부재로 해석하지 않습니다.
- 인용되지 않은 reference나 activity에 연결되지 않는 reference가 있나요? 최종 답변의 인용도 실제 근거 내용과 일치하는지 확인합니다.
- 원문은 출력에서 500자 미리보기로 표시합니다. 필요한 항목만 `v2_process_trace['references'][0]`처럼 로컬에서 펼쳐 볼 수 있습니다. 반환되지 않은 원문·URL은 추가 조회하거나 추측하지 않습니다.

실습 후 커널을 종료해 토큰과 응답을 메모리에서 제거합니다. 노트북 공유 전 M365·내부 문서가 포함된 실행 출력을 검토하거나 지우세요. 이 절은 용량을 자동으로 중지하거나 KB·KS를 변경하지 않습니다.


In [ ]:
v2_quality_question = '이메일에서 품질 불량에 대해서 요청이 온 로트가 있는지 확인해서 ' \
'해당 로트의 품질 이력을 확인해서 어떤 불량이 발생했는지 확인하자. ' \
'그리고 그 불량이 발생한 장비에서 해당 시점에 발생한 장비 센서의 이력을 파악해서 무슨 문제가 발생했을지를 판단하면 좋을 것 같아. ' \
'장비에서 자체적으로 문제가 있었을수도 있지만, 실제로 그 당시 불량 기인이 작업자에서 발생했을 수 있으니 해당 작업지시서도 검토해보고 인수인계시에도 ' \
'특이한 이력이 있었는지 함께 종합적으로 판단해 인사이트를 제공해.'

v2_quality_result = retrieve_v2_trace(QUALITY_KB_NAME, v2_quality_question)

SyntaxError: invalid syntax (3093993445.py, line 1)

In [ ]:
v2_quality_trace = parse_trace(v2_quality_result)
print(format_trace(v2_quality_trace))
display(Markdown('### 최종 답변\n' + v2_quality_trace['answer']))